In [ ]:
import sys
from pathlib import Path

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/jianghongab/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml

if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    if root_dir.parts[-1:] == ('pollen',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    sys.path.append(root_dir)
print(f"Added the following directory to the PYTHONPATH: {root_dir}")
    
# Set the environment variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

Local environment
Added the following directory to the PYTHONPATH: /Users/hongjiang/git/mlfs-book
HopsworksSettings initialized!


# Daily Feature Pipeline for Grass Pollen (Stockholm)

## Sections:
1. Fetch Pollen Data from Pollenrapporten API
2. Insert into Feature Group

**Schedule this notebook to run daily during pollen season (May-August)**

In [10]:
import datetime
import pandas as pd
import hopsworks
from mlfs.airquality import util
import warnings
import json
warnings.filterwarnings("ignore")

## Connect to Hopsworks

In [11]:
project = hopsworks.login(engine="python")
fs = project.get_feature_store()
secrets = hopsworks.get_secrets_api()

location_str = secrets.get_secret("SENSOR_LOCATION_JSON").value
location = json.loads(location_str)
country=location['country']
city=location['city']
street=location['street']

# Stockholm coordinates
latitude = location['latitude']
longitude = location['longitude']

today = datetime.date.today()
print(f"Fetching pollen data for: {today}")

2025-12-31 15:28:09,512 INFO: Closing external client and cleaning up certificates.
2025-12-31 15:28:09,516 INFO: Connection closed.
2025-12-31 15:28:09,517 INFO: Initializing external client
2025-12-31 15:28:09,518 INFO: Base URL: https://c.app.hopsworks.ai:443


2025-12-31 15:28:10,978 INFO: Python Engine initialized.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/1292436
Fetching pollen data for: 2025-12-31


## Get Feature Group Reference

## Fetch Today's Pollen Data

In [13]:
# Fetch last 7 days to ensure we get today's data
start_date = (today - datetime.timedelta(days=7)).strftime('%Y-%m-%d')
end_date = today.strftime('%Y-%m-%d')

pollen_df = util.get_historical_pollen(
    start_date=start_date,
    end_date=end_date
)

if not pollen_df.empty:
    # Get only today's data
    pollen_df['date'] = pd.to_datetime(pollen_df['date']).dt.date
    pollen_today = pollen_df[pollen_df['date'] == today]
    pollen_today['date'] = pd.to_datetime(pollen_today['date'])
    print(f"Pollen data retrieved: {len(pollen_today)} records")
    print(pollen_today)
else:
    print("No pollen data available (likely outside monitoring season)")
    pollen_today = pd.DataFrame()

No pollen data available for 2025-12-24 to 2025-12-31
No pollen data available (likely outside monitoring season)


Get Weather Forecast data

In [28]:
hourly_df = util.get_hourly_weather_forecast(city, latitude, longitude)
hourly_df = hourly_df.set_index('date')

# We will only make 1 daily prediction, so we will replace the hourly forecasts with a single daily forecast
# We only want the daily weather data, so only get weather at 12:00
daily_df = hourly_df.between_time('11:59', '12:01')
daily_df = daily_df.reset_index()
daily_df['date'] = pd.to_datetime(daily_df['date']).dt.date
daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df['city'] = city

daily_df['day_of_year'] = daily_df['date'].dt.dayofyear
daily_df['month'] = daily_df['date'].dt.month
daily_df['is_high_season'] = daily_df['day_of_year'].apply(lambda x: 1 if 140 <= x <= 250 else 0)

# Add GDD (Growing Degree Days) feature: Max(0, (T_mean - T_base))
T_base = 5.0  # Base temperature for grass growth
daily_df['gdd_daily'] = daily_df['temperature_2m_mean'].apply(lambda t: max(0, t - T_base))
daily_df['gdd_cumsum'] = daily_df.groupby(daily_df['date'].dt.year)['gdd_daily'].cumsum()


# Add lagged weather features (yesterday's weather affects today's pollen)
daily_df['precip_lag_1'] = daily_df['precipitation_sum'].shift(1)
daily_df['temp_lag_1'] = daily_df['temperature_2m_mean'].shift(1)
daily_df['wind_lag_1'] = daily_df['wind_speed_10m_max'].shift(1)
daily_df['date'] = pd.to_datetime(daily_df['date']).dt.strftime('%Y-%m-%d %H:%M:%S')
daily_df = daily_df.rename(columns={'date': 'datetime_id'}) # 改名
daily_df

Coordinates 59.25°N 18.0°E
Elevation 24.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s


,datetime_id,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city,day_of_year,month,is_high_season,gdd_daily,gdd_cumsum,precip_lag_1,temp_lag_1,wind_lag_1
0,2025-12-31 00:00:00,-5.20,0.0,5.351785,317.726379,Stockholm,365,12,0,0,0,NaN,NaN,NaN
1,2026-01-01 00:00:00,0.80,2.1,25.864943,124.796097,Stockholm,1,1,0,0,0,0.0,-5.20,5.351785
2,2026-01-02 00:00:00,-1.25,2.0,16.099689,333.435028,Stockholm,2,1,0,0,0,2.1,0.80,25.864943
3,2026-01-03 00:00:00,0.05,0.2,12.864649,17.928020,Stockholm,3,1,0,0,0,2.0,-1.25,16.099689
4,2026-01-04 00:00:00,-2.50,0.0,17.651016,11.768270,Stockholm,4,1,0,0,0,0.2,0.05,12.864649
5,2026-01-05 00:00:00,-6.20,0.0,10.990322,328.392548,Stockholm,5,1,0,0,0,0.0,-2.50,17.651016
6,2026-01-06 00:00:00,-9.45,0.0,6.214563,169.992081,Stockholm,6,1,0,0,0,0.0,-6.20,10.990322


## Upload to Feature Store

In [ ]:
if not pollen_today.empty:
    # grass_pollen_fg.insert(pollen_today, wait=True)
    print("✅ Pollen data uploaded successfully")
else:
    print("⚠️ No data to upload")

⚠️ No data to upload


In [ ]:
# Insert new data
daily_df = daily_df.drop(columns=['day_of_year', 'month', 'is_high_season', 'gdd_daily', 'gdd_cumsum', 'precip_lag_1', 'temp_lag_1','wind_lag_1'], errors='ignore')
# weather_fg.insert(daily_df, wait=True)

2025-12-31 15:49:22,561 INFO: 	2 expectation(s) included in expectation_suite.
Validation succeeded.
Validation Report saved successfully, explore a summary at https://c.app.hopsworks.ai:443/p/1292436/fs/1265790/fg/1880524


Uploading Dataframe: 100.00% |█| Rows 7/7 | Elapsed Time: 00:01 | Remaining Time


Launching job: weather_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1292436/jobs/named/weather_1_offline_fg_materialization/executions
2025-12-31 15:49:39,859 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2025-12-31 15:49:46,231 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2025-12-31 15:51:31,467 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2025-12-31 15:51:31,629 INFO: Waiting for log aggregation to finish.
2025-12-31 15:51:50,389 INFO: Execution finished successfully.


Online data ingestion progress: 0.00% |          | Rows 0/7

(Job('weather_1_offline_fg_materialization', 'SPARK'),
 {
   "success": true,
   "results": [
     {
       "success": true,
       "expectation_config": {
         "expectation_type": "expect_column_min_to_be_between",
         "kwargs": {
           "column": "precipitation_sum",
           "min_value": -0.1,
           "max_value": 1000.0,
           "strict_min": true
         },
         "meta": {
           "expectationId": 801824
         }
       },
       "result": {
         "observed_value": 0.0,
         "element_count": 7,
         "missing_count": null,
         "missing_percent": null
       },
       "meta": {
         "ingestionResult": "INGESTED",
         "validationTime": "2025-12-31T02:49:22.000561Z"
       },
       "exception_info": {
         "raised_exception": false,
         "exception_message": null,
         "exception_traceback": null
       }
     },
     {
       "success": true,
       "expectation_config": {
         "expectation_type": "expect_column_